# Can search signals prioritize content pages for human review?

**Author:** Pratheek Rao  
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring  
**Analysis window:** March 1–31, 2026

## Abstract

Can early search signals help teams decide which content pages to review first? We analyze March 2026 from the FlyRank internship warehouse build v20260703, using 151,981 early-period candidates and an 85,565-page subset with complete later-period measurements. We compare a logistic-regression ranking over five March 1–15 features with the fixed three-signal Week 4 rule on the same client-grouped holdout, where 31 clients train and 8 clients are held out with no overlap. On 4,113 held-out pages, both rankers have Precision@20 of 0.55; the model reaches 0.48 versus 0.46 Precision@50 and 0.767 versus 0.572 ROC-AUC, while the decline-proxy base rate is 0.322. The result supports using the ranking as a human review aid in this historical sample, not predicting future performance or establishing that a refresh improves outcomes.

This notebook mirrors the deployed research page. It reconstructs the March warehouse analysis, compares the learned ranker and Week 4 rule on an identical client-grouped holdout, and regenerates the public-safe figures.

## 1. Introduction / problem statement

**Research question:** Can early search-performance signals help a content team prioritize pages for human review?

The unit is one pseudonymized client/page pair. The output is a rank and suggested action; a reviewer inspects each page in current context. A false positive costs reviewer time, while a missed candidate may leave an opportunity unnoticed. This is a prioritization question, not a causal evaluation of refresh impact.

In [ ]:
from pathlib import Path
import os
from huggingface_hub import hf_hub_download

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Repository root not found")

repo_root = find_repo_root(Path.cwd().resolve())
token = os.getenv("HF_TOKEN")
env_path = next((base / ".env" for base in [repo_root, *repo_root.parents] if (base / ".env").exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith("HF_TOKEN="):
            token = line.split("=", 1)[1].strip().strip(chr(34)).strip(chr(39))
print("Local Hugging Face authentication configured:", bool(token))

## 2. Data

The source is the gated FlyRank internship warehouse, build v20260703, table fact_content_daily_performance, March 2026 partition. Feature window: March 1–15. Label window: March 16–31, requiring all 16 dates.

The early candidate frame has 151,981 pages; 85,565 have complete later measurements; 55,902 have partial later availability and are excluded from target evaluation. The action queue still uses all early candidates. The June _sample and 90-day query table are excluded. IDs are for grouping/ties only, never predictive features. No raw query, URL, client name, token, or row-level export is published.

In [2]:
import duckdb
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

march_url = hf_hub_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet',
    token=token,
)

con = duckdb.connect()
feature_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date <= DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_avg_position * gsc_impressions ELSE 0 END)
      / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15'
                             AND gsc_data_available IS TRUE
                        THEN gsc_impressions ELSE 0 END), 0) AS weighted_avg_position,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15'
                            AND gsc_data_available IS TRUE
                        THEN report_date END) AS active_search_days,
    SUM(CASE WHEN report_date > DATE '2026-03-15'
                  AND gsc_data_available IS TRUE
             THEN gsc_impressions ELSE 0 END) AS late_impressions,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15'
                            AND gsc_data_available IS TRUE
                        THEN report_date END) AS late_available_days
  FROM read_parquet(?)
  GROUP BY client_hash_id, content_hash_id
), early_candidates AS (
  SELECT *,
         CASE WHEN early_impressions > 0
              THEN 100.0 * early_clicks / early_impressions ELSE NULL END AS early_ctr
  FROM page_windows
  WHERE early_impressions > 0
), evaluable AS (
  SELECT *,
         CASE WHEN late_available_days = 16
                   AND late_impressions / 16.0 < early_impressions / 15.0
              THEN 1 ELSE 0 END AS is_declining_proxy
  FROM early_candidates
  WHERE late_available_days = 16
    AND weighted_avg_position IS NOT NULL
    AND early_ctr IS NOT NULL
)
SELECT client_hash_id, content_hash_id, early_impressions, early_clicks,
       weighted_avg_position, active_search_days, early_ctr, is_declining_proxy
FROM evaluable
'''
df = con.execute(feature_query, [march_url]).fetchdf()

# Candidate and evaluability counts are kept in separate SQL-derived stages.
population_query = '''
WITH page_windows AS (
  SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS early_impressions,
    COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS early_available_days,
    COUNT(DISTINCT CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available IS TRUE THEN report_date END) AS late_available_days
  FROM read_parquet(?)
  GROUP BY client_hash_id, content_hash_id
)
SELECT
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0) AS early_candidate_pages,
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0 AND late_available_days = 16) AS evaluable_later_label_pages,
  COUNT(*) FILTER (WHERE early_impressions > 0 AND early_available_days > 0 AND late_available_days >= 1 AND late_available_days < 16) AS partial_late_availability_pages
FROM page_windows
'''
pop_counts = con.execute(population_query, [march_url]).fetchone()
early_candidate_rows, evaluable_later_rows, partial_late_rows = pop_counts
feature_columns = ['early_impressions','early_clicks','weighted_avg_position','active_search_days','early_ctr']
X = df[feature_columns].copy()
y = df['is_declining_proxy'].astype(int)
print('Development month: 2026-03')
print('Early-period candidate pages (early availability only):', early_candidate_rows)
print('Pages with evaluable late label (16 available late dates):', evaluable_later_rows)
print('Pages with partial late availability (1-15 dates):', partial_late_rows)
print('Measured evaluation pages:', len(df))
print('Unique client/page pairs:', df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0])
assert len(df) == df[['client_hash_id','content_hash_id']].drop_duplicates().shape[0]
assert len(df) <= evaluable_later_rows

# Client-grouped split, matching the Week 5 GroupShuffleSplit logic.
groups = df['client_hash_id'].astype(str)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
g_train, g_test = next(gss.split(X, y, groups=groups))

def precision_at_k(frame, probability_column, k):
    top = frame.sort_values(probability_column, ascending=False, kind='mergesort').head(k)
    return float(top['is_declining_proxy'].mean()) if len(top) else 0.0

def fit_and_evaluate(train_pos, test_pos):
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ])
    model.fit(X.iloc[train_pos], y.iloc[train_pos])
    result = df.iloc[test_pos].copy()
    result['model_probability'] = model.predict_proba(X.iloc[test_pos])[:, 1]
    return model, result

model_grouped, grouped_test = fit_and_evaluate(g_train, g_test)
train_df = df.iloc[g_train]
train_clients = set(train_df['client_hash_id'].astype(str))
test_clients = set(grouped_test['client_hash_id'].astype(str))

r_train, r_test = train_test_split(np.arange(len(df)), test_size=0.20, random_state=42, stratify=y)
model_random, random_test = fit_and_evaluate(r_train, r_test)
random_train_clients = set(df.iloc[r_train]['client_hash_id'].astype(str))
random_test_clients = set(random_test['client_hash_id'].astype(str))

comparison = pd.DataFrame([
    {
        'Split':'Client-grouped (honest reference)','Train pages':len(g_train),'Test pages':len(g_test),
        'Train clients':len(train_clients),'Test clients':len(test_clients),'Client overlap':len(train_clients.intersection(test_clients)),
        'Precision@20':precision_at_k(grouped_test,'model_probability',20),'Precision@50':precision_at_k(grouped_test,'model_probability',50),
        'ROC-AUC':roc_auc_score(grouped_test['is_declining_proxy'], grouped_test['model_probability']),
        'Positive-class/base rate':float(grouped_test['is_declining_proxy'].mean()),
    },
    {
        'Split':'Random row split (diagnostic only)','Train pages':len(r_train),'Test pages':len(r_test),
        'Train clients':len(random_train_clients),'Test clients':len(random_test_clients),'Client overlap':len(random_train_clients.intersection(random_test_clients)),
        'Precision@20':precision_at_k(random_test,'model_probability',20),'Precision@50':precision_at_k(random_test,'model_probability',50),
        'ROC-AUC':roc_auc_score(random_test['is_declining_proxy'], random_test['model_probability']),
        'Positive-class/base rate':float(random_test['is_declining_proxy'].mean()),
    },
])
print(comparison.to_string(index=False, float_format=lambda x:f'{x:.4f}'))
print('Metrics apply only to the evaluable 16-date late-period subset:', len(df))
assert len(train_clients.intersection(test_clients)) == 0
assert grouped_test['is_declining_proxy'].nunique() == 2
assert random_test['is_declining_proxy'].nunique() == 2


# Compute the unchanged Week 4 rule on the exact client-grouped test rows.
grouped_test['baseline_score'] = (
    (grouped_test['early_impressions'] >= 500).astype(int)
    + ((grouped_test['weighted_avg_position'] > 0) & (grouped_test['weighted_avg_position'] <= 20)).astype(int)
    + (grouped_test['early_ctr'] < 0.5).astype(int)
)
grouped_test['active_search_days'] = df.iloc[g_test]['active_search_days'].values
baseline_ranked = grouped_test.sort_values(
    ['baseline_score','active_search_days','early_impressions','client_hash_id','content_hash_id'],
    ascending=[False,False,False,True,True],kind='mergesort'
)
model_ranked = grouped_test.sort_values(
    ['model_probability','client_hash_id','content_hash_id'],
    ascending=[False,True,True],kind='mergesort'
)
def p_at(frame,k):
    return float(frame.head(k)['is_declining_proxy'].mean())
base_rate=float(grouped_test['is_declining_proxy'].mean())
baseline_auc=float(roc_auc_score(grouped_test['is_declining_proxy'],grouped_test['baseline_score']))
model_auc=float(roc_auc_score(grouped_test['is_declining_proxy'],grouped_test['model_probability']))
model_baseline_comparison=pd.DataFrame([
    {'Method':'Week 4 rule baseline','Precision@20':p_at(baseline_ranked,20),'Precision@50':p_at(baseline_ranked,50),'ROC-AUC':baseline_auc,'Base rate':base_rate},
    {'Method':'Logistic regression','Precision@20':p_at(model_ranked,20),'Precision@50':p_at(model_ranked,50),'ROC-AUC':model_auc,'Base rate':base_rate},
])
print('Same held-out pages:',len(grouped_test),'| train clients:',len(train_clients),'| test clients:',len(test_clients),'| overlap:',len(train_clients & test_clients))
print(model_baseline_comparison.to_string(index=False,float_format=lambda x:f'{x:.4f}'))

Development month: 2026-03
Early-period candidate pages (early availability only): 151981
Pages with evaluable late label (16 available late dates): 85565
Pages with partial late availability (1-15 dates): 55902
Measured evaluation pages: 85565
Unique client/page pairs: 85565
                             Split  Train pages  Test pages  Train clients  Test clients  Client overlap  Precision@20  Precision@50  ROC-AUC  Positive-class/base rate
 Client-grouped (honest reference)        81452        4113             31             8               0        0.5500        0.4800   0.7667                    0.3221
Random row split (diagnostic only)        68452       17113             39            35              35        0.8500        0.7200   0.7056                    0.4183
Metrics apply only to the evaluable 16-date late-period subset: 85565
Same held-out pages: 4113 | train clients: 31 | test clients: 8 | overlap: 0
              Method  Precision@20  Precision@50  ROC-AUC  Base rate
Wee

## 3. Methodology

Five prediction-time features: early impressions, early clicks, impression-weighted average position, count of available search days, and early CTR as a percentage. The proxy label is 1 if average daily impressions fall from March 1–15 to March 16–31, with the full 16 late dates observed.

The baseline scores the three Week 4 conditions: impressions ≥500, weighted position in (0,20], and CTR <0.5%; score 3 suggests human CTR review. The model is standardized class-balanced logistic regression (max_iter=1000, seed 42). Validation uses GroupShuffleSplit(test_size=0.20, random_state=42) by client. Model and rule use the same held-out rows and labels; preprocessing is fitted on training pages only. Future, target-derived, ID, and action/product fields are excluded.

In [3]:
# Verified validation totals and leakage boundaries.
assert len(df)==85565 and len(grouped_test)==4113
assert len(train_clients)==31 and len(test_clients)==8 and len(train_clients & test_clients)==0
assert len(feature_columns)==5
assert set(feature_columns).isdisjoint({'late_impressions','late_clicks','is_declining_proxy','trend_direction','trend_pct','client_hash_id','content_hash_id'})
print('Evaluable pages:',len(df),'| train pages:',len(g_train),'| holdout pages:',len(g_test))
print('Feature count:',len(feature_columns),'| client overlap: 0 | leakage check: PASS')

Evaluable pages: 85565 | train pages: 81452 | holdout pages: 4113
Feature count: 5 | client overlap: 0 | leakage check: PASS


## 4. Results (model versus baseline)

On the shared grouped holdout, the model ties the transparent rule at Precision@20, has a two-page advantage at Precision@50 (24/50 vs 23/50), and has higher ROC-AUC in this one split. Base rate is reported alongside each method. The test set includes eight held-out clients, so do not generalize beyond the observed sample.

In [4]:
import json
import shutil
# Save only aggregated results and regenerate publication-safe figures.
from html import escape
metrics={
 'release':'FlyRank internship warehouse build v20260703','table':'fact_content_daily_performance','month':'2026-03',
 'early_candidate_pages':int(early_candidate_rows),'complete_label_pages':int(len(df)),
 'train_pages':int(len(g_train)),'test_pages':int(len(g_test)),
 'train_clients':int(len(train_clients)),'test_clients':int(len(test_clients)),
 'client_overlap':int(len(train_clients & test_clients)),'positive_class_base_rate':base_rate,'seed':42,
 'methods':model_baseline_comparison.to_dict(orient='records'),
 'features':feature_columns,'label':'late mean daily impressions below early mean daily impressions; complete 16-date late window',
 'public_safety':'aggregate results only; no IDs, raw queries, URLs, names, tokens, or row-level records'
}
out=repo_root/'work/outputs/capstone_metrics.json'; out.write_text(json.dumps(metrics,indent=2)+'\n')
assets=repo_root/'paper/assets'; assets.mkdir(parents=True,exist_ok=True)
shutil.copy2(repo_root/'work/figures/w07_action_distribution.svg',assets/'w07_action_distribution.svg')
shutil.copy2(repo_root/'work/figures/w07_decay_proxy_by_position.svg',assets/'w07_decay_proxy_by_position.svg')
colors=['#c98535','#245848']; rows=[('Precision@20','Precision@20'),('Precision@50','Precision@50'),('ROC-AUC','ROC-AUC')]
svg=['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 820 420" role="img" aria-labelledby="title desc">','<title id="title">Grouped holdout ranking metrics</title>','<desc id="desc">Week 4 rule and logistic regression on identical held-out pages. Base rate 0.322.</desc>','<rect width="820" height="420" fill="#fffefa"/>','<text x="30" y="40" font-family="Arial,sans-serif" font-size="21" font-weight="700">Ranking results on the same client holdout</text>',f'<text x="30" y="65" font-family="Arial,sans-serif" font-size="13">{len(grouped_test):,} pages · {len(test_clients)} held-out clients · positive rate {base_rate:.3f}</text>']
for j,(label,key) in enumerate(rows):
 y0=112+j*94; svg.append(f'<text x="30" y="{y0}" font-family="Arial,sans-serif" font-size="15" font-weight="700">{label}</text>')
 if j<2: svg.append(f'<line x1="{290+440*base_rate:.1f}" y1="{y0+7}" x2="{290+440*base_rate:.1f}" y2="{y0+68}" stroke="#3b6d8c" stroke-width="2" stroke-dasharray="5,4"/>')
 for i,method in enumerate(['Week 4 rule baseline','Logistic regression']):
  val=float(model_baseline_comparison.iloc[i][key]); y=y0+13+i*28; w=440*val
  svg.append(f'<text x="30" y="{y+13}" font-family="Arial,sans-serif" font-size="12">{method}</text><rect x="290" y="{y}" width="{w:.1f}" height="18" rx="3" fill="{colors[i]}"/><text x="{300+w:.1f}" y="{y+14}" font-family="Arial,sans-serif" font-size="12">{val:.3f}</text>')
 svg.append(f'<line x1="290" y1="{y0+72}" x2="730" y2="{y0+72}" stroke="#d9d8ce"/>')
svg+=['<text x="30" y="398" font-family="Arial,sans-serif" font-size="12">Dashed markers in precision rows = positive-class base rate</text>','</svg>']
(assets/'heldout_comparison.svg').write_text('\n'.join(svg)+'\n')
assert round(p_at(baseline_ranked,20),2)==0.55 and round(p_at(model_ranked,20),2)==0.55
assert round(p_at(baseline_ranked,50),2)==0.46 and round(p_at(model_ranked,50),2)==0.48
assert round(baseline_auc,3)==0.572 and round(model_auc,3)==0.767
print('Aggregate receipt and three figures generated; result assertions passed.')

Aggregate receipt and three figures generated; result assertions passed.


## 5. Limitations & honest framing

The label is historical, not a future forecast or refresh-effect estimate. Only eight clients are held out; one split does not show all uncertainty. Complete-label pages may differ from pages with incomplete later measurements. Seasonality, competition, and indexing are not captured. The rule queue uses all early candidates while model metrics apply to the evaluable held-out subset. No reviewer acceptance or monetary cost data was available.

In [5]:
# Check the Week 7 action population used by these recommendations.
import json
w07_path = repo_root / "work/outputs/w07_action_playbook_metrics.json"
w07 = json.loads(w07_path.read_text(encoding="utf-8"))
queue = w07["population"]["full_early_period_action_queue"]
actions = queue["action_distribution"]
assert queue["candidate_pages"] == 151981
assert actions["review_ctr_refresh"] == 26597
assert actions["monitor"] == 125384
print("Early candidates:", f'{queue["candidate_pages"]:,}')
print("Suggested human review:", f'{actions["review_ctr_refresh"]:,}')
print("Monitor:", f'{actions["monitor"]:,}')

Early candidates: 151,981
Suggested human review: 26,597
Monitor: 125,384


## 7. Reproducibility

The notebook reconstructs the March frame from the gated parquet partition, applies the fixed client-grouped split (seed 42), compares both methods on identical rows, and writes aggregate metrics to `work/outputs/capstone_metrics.json`. It regenerates three SVG figures under `paper/assets/`. A fresh run requires access to FlyRank/internship-warehouse and a locally configured HF_TOKEN; the token is never printed or saved. Run with `pip install -r requirements.txt`, then open this notebook in Jupyter.

In [6]:
# Confirm generated paper assets and aggregate receipt.
assert (repo_root/'work/outputs/capstone_metrics.json').exists()
assert all((repo_root/'paper/assets'/name).exists() for name in ['heldout_comparison.svg','w07_action_distribution.svg','w07_decay_proxy_by_position.svg'])
print('Aggregate metrics receipt: PASS | paper figure assets: PASS')

Aggregate metrics receipt: PASS | paper figure assets: PASS


## 8. Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

In [7]:
# Final public-safety checks for the served folder.
import re
served_files = [p for p in (repo_root / "paper").rglob("*") if p.is_file()]
assert all(p.suffix in {".html", ".svg"} for p in served_files)
public_text = chr(10).join(p.read_text(encoding="utf-8") for p in served_files)
assert not re.search(r"client_[0-9a-f]{8,}|content_[0-9a-f]{8,}", public_text, re.I)
assert not re.search(r"hf_[A-Za-z0-9]{20,}", public_text)
assert "https://flyrank.ai" in public_text
assert "github.com/pratheekrao12" in public_text
print("Served folder contains only HTML/SVG: PASS")
print("No pseudonymous row IDs or token-shaped strings: PASS")
print("Published links are intended public sources: PASS")

Served folder contains only HTML/SVG: PASS
No pseudonymous row IDs or token-shaped strings: PASS
Published links are intended public sources: PASS


## Self-check

- [x] Five-sentence abstract and every required paper section are present.
- [x] Model and baseline use the same client-grouped holdout.
- [x] The title, tables, and captions distinguish historical association from causal or future claims.
- [x] Three aggregate figures and the FlyRank acknowledgment are present.
- [x] The publication folder contains only the intended paper page and figures.

The executable assertions in the previous cells are the final check.